In [ ]:
import os
import pandas as pd
from collections import Counter


# 데이터 로드

In [ ]:
# 형태소 처리된 비공개 분석 입력 위치
file_path = os.environ['LG_VOC_DATA_DIR'] + os.sep

styler = pd.read_csv(file_path + '의류관리기_수정.csv', encoding='utf-8-sig')
vacuum = pd.read_csv(file_path + '무선스틱청소기_수정.csv', encoding='utf-8-sig')


## 불용어 처리

In [ ]:
### 불용어 처리 리스트
stop_words = {'이다', '에서', '후', '번', '일', '랑', '보다', '를', '와', '내', '과', '되다', '전자서비스센터',
              '시', '때', '데', '을', '는', '가', '이', '개', '때문', '꼭', '경우', '은', '에도',
              '되어다', '수', '의', '없다', '나', '까지', '이제', '같다', '뿐', '해주다', '않다',
              '다이렉트결혼준비', '에', '전', '들', '중', '다', '기', '그렇다', '니', '도', '등', '있다',
              '분', '더', '정말', '습', '게', '하다', '적', '아니다', '매', '더욱', '슈', '인', '안', '해', 'S',
              '만', '바로', '로', '한', 'CB', '처럼', '니다', '그', '오브', '그리고', '통해', '점', '제', '이나',
              '한번', '글', '체험단', '요', '으로', '저희', '거', '하고', '것', '부터', '에는', '기다', '저',
              '블랙틴트미러', '그래서', '특히', '예요', '또한', '많이', '없이', '호텔', '수업', '교시', '출판',
              '서점', '이벤트', '발매', '돼다', '보시', '이라', '참고', '오다', '맞다', '같이', '근데', '그냥', '약간',
              '다음', '여기', '조금', '수가', '하나', '대해', '어떻다', '싶다', '이라고', '너무', '정도', '자주', '이렇게',
              '인데', '이기', '가장', '하지만', '해당', '한다는', '사하', '가장', '는걸', '아주', '고', '\n', '\xa0',
              ' ', '레몬테라스카페', '즉', '다음', '동영상', '취소', '재생', '속도', '음소거', '음소거설정', '실시간', '설정',
              '실시간설정', '다시시도',
              '골프', '레슨', '도움말', '도움말상태', '라이센스', '고화질', '영상', '해상도', '문제', '발생',
              '해상도', '자막', '안함', '초', '광고', '공유', '전체화면', '자막안함옵션',
              '글자', '크기배경색', '도움말', '상태', '배경색기본', '골프연습장', '회원님',
              '골프레슨', '죄송하다', '문제발생', '가능한영상', '자막안함', '글자크기',
              '범례부동산거리', '전체화면', '크기배경색','옵션', '기본자동옵션', '시가총액', '억원', '상승',
              '차량용', '차량', '용', '원', '현재', '동시', '군', '구시', '거리', '읍', '면', '도국', '지도',
              '범례', '부동산', '거리', '데이터', '다시시도', '존', '인', '피트', '회원님', '들다', '기본옵션', '오후', '우리집',
              '년', '집', '및', '준비물', '다시', '시도', '로켓', '배송'
              }

In [ ]:
def remove_stopwords_from_df(df, columns, stop_words):
    # 불용어 처리 함수
    def remove_stopwords(text, stop_words):
        if not isinstance(text, str):
            return ''
        
        # 띄어쓰기를 기준으로 단어 분리
        tokens = text.split()
        
        # 불용어 제거
        filtered_tokens = [word for word in tokens if word not in stop_words]
        
        # 단어를 다시 문장으로 합치기
        return ' '.join(filtered_tokens)

    # 지정된 열에 대해 불용어 처리
    for col in columns:
        if col in df.columns:
            df[col] = df[col].apply(lambda x: remove_stopwords(x, stop_words))
    
    return df

styler = remove_stopwords_from_df(styler, ['noun', 'verb_adj', 'nva', 'token'], stop_words)
vacuum = remove_stopwords_from_df(vacuum, ['noun', 'verb_adj', 'nva', 'token'], stop_words)


## 연도/월별 열 추가

In [ ]:
# year와 month 열 추가 함수
def add_year_month_columns(df):
    df['year'] = pd.to_datetime(df['date']).dt.year
    df['month'] = pd.to_datetime(df['date']).dt.month

# 각 데이터프레임에 함수 적용
add_year_month_columns(styler)
add_year_month_columns(vacuum)


# 전체빈도/연도별빈도: NVA

## 전체빈도

In [ ]:
import pandas as pd
from collections import Counter

def count_words(text_series, year_series, top_n=None, year1=None, year2=None):
    # 유효한 데이터로 필터링
    valid_data = pd.DataFrame({'text': text_series, 'year': year_series}).dropna()
    valid_data['text'] = valid_data['text'].astype(str)

    # 전체 텍스트 합치기
    all_text = ' '.join(valid_data['text'])

    # 단어 빈도 계산
    word_counts = Counter(all_text.split())
    total_word_count = sum(word_counts.values()) or 1  # 분모 0 방지

    # 단어 비율 계산
    words_list, freqs = zip(*word_counts.most_common(top_n)) if word_counts else ([], [])
    ratios = [freq / total_word_count for freq in freqs]

    # transition 계산
    transitions = []
    if year1 is not None and year2 is not None:
        # 연도별 데이터 필터링
        year1_texts = valid_data[valid_data['year'] == year1]['text'].dropna().astype(str)
        year2_texts = valid_data[valid_data['year'] == year2]['text'].dropna().astype(str)

        year1_words = ' '.join(year1_texts).split()
        year2_words = ' '.join(year2_texts).split()

        year1_counts = Counter(year1_words)
        year2_counts = Counter(year2_words)

        total1 = sum(year1_counts.values()) or 1  # 분모 0 방지
        total2 = sum(year2_counts.values()) or 1

        for word in words_list:
            count1 = year1_counts.get(word, 0)
            count2 = year2_counts.get(word, 0)

            # transition 계산: 변화율 (year2 빈도 - year1 빈도) / year1 빈도 * 100
            if count1 > 0:
                transition_rate = ((count2/total2 - count1/total1) / (count1/total1)) * 100
            else:
                transition_rate = 0  # year1의 빈도가 0이면 변화율을 0으로 설정

            transitions.append(transition_rate)
    else:
        transitions = [0] * len(words_list)  # 연도별 데이터가 없는 경우 0으로 채움

    # 결과 데이터프레임 생성
    result_df = pd.DataFrame({
        'word': words_list,
        'freq': freqs,
        'ratio': ratios,
        'transition': transitions
    })

    # 빈도(freq) 기준으로 순위를 매기고 rank 열 추가
    result_df['rank'] = result_df['freq'].rank(method='dense', ascending=False).astype(int)

    # rank 열을 가장 왼쪽으로 이동
    result_df = result_df[['rank', 'word', 'freq', 'ratio', 'transition']]

    return result_df


In [ ]:
styler_total_counts = count_words(text_series=styler['nva'], year_series=styler['year'], top_n=50)
vacuum_total_counts = count_words(text_series=vacuum['nva'], year_series=vacuum['year'], top_n=50)


## 연도별 빈도

In [ ]:
import pandas as pd
from collections import Counter

def count_words_by_year(text_series, year_series, year_range, top_n=None):
    # 유효한 데이터로 필터링
    valid_data = pd.DataFrame({'text': text_series, 'year': year_series}).dropna()
    valid_data['text'] = valid_data['text'].astype(str)

    # 연도 범위 필터링
    valid_data = valid_data[valid_data['year'].isin(year_range)]
    unique_years = sorted(valid_data['year'].unique())

    # 결과 저장용 리스트 초기화
    result_data = []

    # 이전 연도 단어 빈도 초기화
    prev_year_counts = Counter()

    for i, year in enumerate(unique_years):
        # 현재 연도의 텍스트 데이터
        current_texts = valid_data[valid_data['year'] == year]['text']
        all_text = ' '.join(current_texts)

        # 단어 빈도 계산
        word_counts = Counter(all_text.split())
        total_word_count = sum(word_counts.values()) or 1  # 분모 0 방지

        # 단어 비율 계산 및 transition 계산
        words_list, freqs = zip(*word_counts.most_common(top_n)) if word_counts else ([], [])
        ratios = [freq / total_word_count for freq in freqs]

        transitions = []
        if i == 0:  # 첫 번째 연도는 변화율 계산하지 않음
            transitions = [0] * len(words_list)
        else:
            for word in words_list:
                prev_freq = prev_year_counts.get(word, 0)
                prev_total = sum(prev_year_counts.values()) or 1
                prev_ratio = prev_freq / prev_total
                current_ratio = word_counts[word] / total_word_count

                # 변화율 계산
                transition = (current_ratio - prev_ratio) / prev_ratio * 100 if prev_ratio > 0 else 0
                transitions.append(transition)

        # 현재 연도의 결과를 리스트에 저장
        for word, freq, ratio, transition in zip(words_list, freqs, ratios, transitions):
            result_data.append([word, year, freq, ratio, transition])

        # 현재 연도의 단어 빈도를 이전 연도로 설정
        prev_year_counts = word_counts

    # 결과 데이터를 데이터프레임으로 변환
    result_df = pd.DataFrame(result_data, columns=['word', 'year', 'freq', 'ratio', 'transition'])

    # 같은 단어를 기준으로 연도별 빈도와 비율 정렬
    all_words = sorted(result_df['word'].unique())
    final_result = pd.DataFrame({'word': all_words})
    for year in year_range:
        year_data = result_df[result_df['year'] == year][['word', 'freq', 'ratio', 'transition']]
        year_data = year_data.rename(columns={
            'freq': f'freq_{year}',
            'ratio': f'ratio_{year}',
            'transition': f'transition_{year}'
        })
        final_result = final_result.merge(year_data, on='word', how='left')
        final_result[f'word_{year}'] = final_result['word'].where(final_result[f'freq_{year}'].notna())

    # 전체 기간 빈도를 합산해 순위 계산
    freq_columns = [f'freq_{year}' for year in year_range]
    final_result.insert(0, 'rank', final_result[freq_columns].fillna(0).sum(axis=1).rank(method='dense', ascending=False).astype(int))
    final_result = final_result.sort_values('rank').reset_index(drop=True)
    return final_result


In [ ]:
year_range = [2022, 2023, 2024]

styler_total_counts_years = count_words_by_year(text_series=styler['nva'], year_series=styler['year'], year_range=year_range, top_n=50)
vacuum_total_counts_years = count_words_by_year(text_series=vacuum['nva'], year_series=vacuum['year'], year_range=year_range, top_n=50)


## csv 저장

In [ ]:
# CSV로 저장하기
total_counts_path = os.path.join("results", "빈도분석") + os.sep
os.makedirs(total_counts_path, exist_ok=True)

styler_total_counts.to_csv(total_counts_path + 'styler_total_counts.csv', index = False, encoding='utf-8-sig')
vacuum_total_counts.to_csv(total_counts_path + 'vacuum_total_counts.csv', index = False, encoding='utf-8-sig')
styler_total_counts_years.to_csv(total_counts_path + 'styler_total_counts_years.csv', index=False, encoding="utf-8-sig")
vacuum_total_counts_years.to_csv(total_counts_path + 'vacuum_total_counts_years.csv', index=False, encoding="utf-8-sig")


## 증감 check!

### 함수

In [ ]:
# 1. 2022 -> 2023 -> 2024 transition 값이 모두 증가한 단어를 확인
def consistently_increasing_words(df):
    increasing_words = []
    
    # 각 단어 확인
    for word in df['word_2022']:
        if pd.isna(word):
            continue
        # 2023 데이터에서 해당 단어 확인
        if word in df['word_2023'].values:
            transition_2023 = df[df['word_2023'] == word]['transition_2023'].values[0]
            
            # transition_2023 값이 양수인 경우에만 계속 진행
            if transition_2023 > 0:
                # 2024 데이터에서 해당 단어 확인
                if word in df['word_2024'].values:
                    transition_2024 = df[df['word_2024'] == word]['transition_2024'].values[0]
                    
                    # transition_2024 값이 양수인 경우 단어 추가
                    if transition_2024 > 0:
                        increasing_words.append(word)
    
    return increasing_words


# 1. 2022 -> 2023 -> 2024 transition 값이 모두 감소한 단어를 확인
def consistently_decreasing_words(df):
    decreasing_words = []
    
    # 각 단어 확인
    for word in df['word_2022']:
        if pd.isna(word):
            continue
        # 2023 데이터에서 해당 단어 확인
        if word in df['word_2023'].values:
            transition_2023 = df[df['word_2023'] == word]['transition_2023'].values[0]
            
            # transition_2023 값이 음수인 경우에만 계속 진행
            if transition_2023 < 0:
                # 2024 데이터에서 해당 단어 확인
                if word in df['word_2024'].values:
                    transition_2024 = df[df['word_2024'] == word]['transition_2024'].values[0]
                    
                    # transition_2024 값이 음수인 경우 단어 추가
                    if transition_2024 < 0:
                        decreasing_words.append(word)
    
    return decreasing_words


# 1. 2022 -> 2023에는 감소, 2023 -> 2024에는 transition 값이 증가한 단어를 확인
# 2. 2022에는 존재하는 단어지만 2023에는 사라진 단어. 2024에 다시 등장 & transition 값이 증가(양수)한 단어
def recently_increasing_words(df):
    recently_increasing_words = []
    
    # 각 단어 확인
    for word in df['word_2022']:
        if pd.isna(word):
            continue
        # 2023 데이터에서 해당 단어 확인
        if word in df['word_2023'].values:
            transition_2023 = df[df['word_2023'] == word]['transition_2023'].values[0]
            
            # transition_2023 값이 음수인 경우에만 계속 진행
            if transition_2023 < 0:
                # 2024 데이터에서 해당 단어 확인
                if word in df['word_2024'].values:
                    transition_2024 = df[df['word_2024'] == word]['transition_2024'].values[0]
                    
                    # transition_2024 값이 양수인 경우 단어 추가
                    if transition_2024 > 0:
                        recently_increasing_words.append(word)
    
    # 조건 2
    for word in df['word_2022']:
        if pd.isna(word):
            continue
        # 2023 데이터에서 해당 단어 없는거 확인
        if word not in df['word_2023'].values:
                # 2024 데이터에서 해당 단어 확인
                if word in df['word_2024'].values:
                    transition_2024 = df[df['word_2024'] == word]['transition_2024'].values[0]
                    
                    # transition_2024 값이 양수인 경우 단어 추가
                    if transition_2024 > 0:
                        recently_increasing_words.append(word)

    # 조건 3
    for word in df['word_2023']:
        if pd.isna(word):
            continue
        # transition_2023 값이 양수인 경우에만 계속 진행
        transition_2023 = df[df['word_2023'] == word]['transition_2023'].values[0]
        if transition_2023 > 0:
            # 2022 데이터에서 해당 단어 없는거 확인
            if word not in df['word_2022'].values:
                # 2024 데이터에서 해당 단어 확인
                if word in df['word_2024'].values:
                    transition_2024 = df[df['word_2024'] == word]['transition_2024'].values[0]
                    
                    # transition_2024 값이 양수인 경우 단어 추가
                    if transition_2024 > 0:
                        recently_increasing_words.append(word)

    return recently_increasing_words


# 1. 2022 -> 2023에는 증가, 2023 -> 2024에는 transition 값이 감소한 단어를 확인
def recently_decreasing_words(df):
    recently_decreasing_words = []
    
    # 각 단어 확인
    for word in df['word_2022']:
        if pd.isna(word):
            continue
        # 2023 데이터에서 해당 단어 확인
        if word in df['word_2023'].values:
            transition_2023 = df[df['word_2023'] == word]['transition_2023'].values[0]
            
            # transition_2023 값이 양수인 경우에만 계속 진행
            if transition_2023 > 0:
                # 2024 데이터에서 해당 단어 확인
                if word in df['word_2024'].values:
                    transition_2024 = df[df['word_2024'] == word]['transition_2024'].values[0]
                    
                    # transition_2024 값이 음수인 경우 단어 추가
                    if transition_2024 < 0:
                        recently_decreasing_words.append(word)

    # 조건 2
    for word in df['word_2022']:
        if pd.isna(word):
            continue
        # 2023 데이터에서 해당 단어 없는거 확인
        if word not in df['word_2023'].values:
                # 2024 데이터에서 해당 단어 확인
                if word in df['word_2024'].values:
                    transition_2024 = df[df['word_2024'] == word]['transition_2024'].values[0]
                    
                    # transition_2024 값이 음수인 경우 단어 추가
                    if transition_2024 < 0:
                        recently_decreasing_words.append(word)

    # 조건 3
    for word in df['word_2023']:
        if pd.isna(word):
            continue
        # transition_2023 값이 음수인 경우에만 계속 진행
        transition_2023 = df[df['word_2023'] == word]['transition_2023'].values[0]
        if transition_2023 < 0:
            # 2022 데이터에서 해당 단어 없는거 확인
            if word not in df['word_2022'].values:
                # 2024 데이터에서 해당 단어 확인
                if word in df['word_2024'].values:
                    transition_2024 = df[df['word_2024'] == word]['transition_2024'].values[0]
                    
                    # transition_2024 값이 음수인 경우 단어 추가
                    if transition_2024 < 0:
                        recently_decreasing_words.append(word)

    return recently_decreasing_words


# 1. 2024년 상위 목록에 새로 등장한 단어를 확인
def recently_appeared_words(df):
    recently_appeared_words = []
                          
    # word_2022, word_2023에는 없지만 word_2024에만 존재하는 단어 확인
    for word in df['word_2024']:
        if pd.isna(word):
            continue
        if word not in df['word_2022'].values and word not in df['word_2023'].values:
            recently_appeared_words.append(word)


    return recently_appeared_words



### 실행

In [ ]:
# df = styler_winter_total_counts_years
# df = styler_fall_total_counts_years
# df = vacuum_winter_total_counts_years

# df = styler_total_counts_years
df = vacuum_total_counts_years


print("2022 -> 2023 -> 2024 transition 값이 모두 증가한 단어:")
print(consistently_increasing_words(df))
print("2022 -> 2023 -> 2024 transition 값이 모두 감소한 단어:")
print(consistently_decreasing_words(df))
print("2023 -> 2024 transition 값이 증가한 단어:")
print(recently_increasing_words(df))
print("2023 -> 2024 transition 값이 감소한 단어:")
print(recently_decreasing_words(df))
print("2024년 상위 목록에 새로 등장한 단어:")
print(recently_appeared_words(df))

## 계절별 빈도

In [ ]:
# 계절별 나눔

# 추가 불용어
season_stop_words = [
    "로라스타", "바니에르", "에어베리", "시티파이", "에어락", "오브제컬렉션", "메이트",
    "건조기", "향기", "소형", "미니", "옷", "관리", "시트", "옷장", "제품", "추천",
    "젯", "용"
]

def filter_by_month(df, start_month, end_month, stop_words):
    # 월 범위에 맞는 데이터와 불용어 처리
    filtered_df = df[(df['month'] >= start_month) & (df['month'] <= end_month)].copy()
    filtered_df['nva'] = filtered_df['nva'].fillna('').astype(str).apply(
        lambda x: ' '.join([word for word in x.split() if word not in stop_words])
    )
    return filtered_df

styler_winter = pd.concat([filter_by_month(styler, 1, 2, season_stop_words),
                           filter_by_month(styler, 12, 12, season_stop_words)], ignore_index=True)
styler_fall = filter_by_month(styler, 9, 11, season_stop_words)
vacuum_winter = pd.concat([filter_by_month(vacuum, 1, 2, season_stop_words),
                           filter_by_month(vacuum, 12, 12, season_stop_words)], ignore_index=True)

styler_winter_total_counts_years = count_words_by_year(styler_winter['nva'], styler_winter['year'], year_range, top_n=50)
styler_fall_total_counts_years = count_words_by_year(styler_fall['nva'], styler_fall['year'], year_range, top_n=50)
vacuum_winter_total_counts_years = count_words_by_year(vacuum_winter['nva'], vacuum_winter['year'], year_range, top_n=50)

styler_winter_total_counts_years.to_csv(total_counts_path + 'styler_winter_total_counts_years.csv', index=False, encoding='utf-8-sig')
styler_fall_total_counts_years.to_csv(total_counts_path + 'styler_fall_total_counts_years.csv', index=False, encoding='utf-8-sig')
vacuum_winter_total_counts_years.to_csv(total_counts_path + 'vacuum_winter_total_counts_years.csv', index=False, encoding='utf-8-sig')
